# LightGBM Predict Contrib for Multi-Class Classification

## Class `ShapExplain`

```python

import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import os

class ShapExplain:
    def __init__(self, model):
        if hasattr(model, 'classes_') and hasattr(model, 'feature_name_'):
            self.model = model
            self.raw_scores = None
            self.proba_scores = None
            self.labels = None
            self.shap_values = None
            self.shap_contributions = None

            # Initialize class attributes from the model
            self.num_classes = len(model.classes_)
            self.num_features = len(model.feature_name_)
            self.feature_names = model.feature_name_ + ["Bias"]
        else:
            raise ValueError("The provided model does not have the required attributes 'classes_' or 'feature_name_'.")

    def predict_raw_scores(self, X):
        if hasattr(self.model, 'predict'):
            self.raw_scores = self.model.predict(X, raw_score=True)
            return self.raw_scores
        else:
            raise AttributeError("The model does not support raw score predictions.")

    def predict_proba(self, X):
        if hasattr(self.model, 'predict_proba'):
            self.proba_scores = self.model.predict_proba(X)
            return self.proba_scores
        else:
            raise AttributeError("The model does not support probability predictions.")

    def predict_labels(self, X):
        self.predict_proba(X)  # Ensure probabilities are always calculated
        self.labels = [np.argmax(score) for score in self.proba_scores]
        return self.labels

    def calculate_shap_values(self, X):
        if hasattr(self.model, 'predict'):
            self.shap_values = self.model.predict(X, pred_contrib=True)
            return self.shap_values
        else:
            raise AttributeError("The model does not support SHAP value predictions.")

    def split_shap_values(self):
        if self.shap_values is None:
            raise ValueError("SHAP values have not been calculated. Call calculate_shap_values first.")
        
        shap_contributions = {}
        for class_idx in range(self.num_classes):
            start_idx = class_idx * (self.num_features + 1)
            end_idx = start_idx + (self.num_features + 1)
            shap_contributions[f"Class_{class_idx}"] = pd.DataFrame(
                self.shap_values[:, start_idx:end_idx], columns=self.feature_names
            )
        self.shap_contributions = shap_contributions
        return self.shap_contributions
    
    def calculate_mean_absolute_shap_contributions(self):
        """
        Calculate the mean absolute SHAP contributions for each feature and class.

        Returns:
            pd.DataFrame: A DataFrame where rows represent features, columns represent classes,
                        and values are the mean absolute SHAP contributions.
        """
        if self.shap_contributions is None:
            raise ValueError("SHAP contributions have not been split. Call split_shap_values first.")
        
        mean_abs_contributions = {}
        for class_name, shap_values_df in self.shap_contributions.items():
            # Exclude the "Bias" column and calculate the mean absolute value
            mean_abs_contributions[class_name] = shap_values_df.iloc[:, :-1].abs().mean(axis=0)
        
        # Combine into a DataFrame for easier interpretation
        mean_abs_df = pd.DataFrame(mean_abs_contributions)
        mean_abs_df.index.name = "Feature"
        
        return mean_abs_df

    def plot_shap_summary(self, plot_type='bar', output_dir=None, title_prefix=None):
        
        # Ensure output directory exists if specified
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)

        for class_name, shap_values_df in self.shap_contributions.items():
            shap_values = shap_values_df.iloc[:, :-1].values
            feature_names = shap_values_df.columns[:-1]  # Exclude 'Bias' from feature names

            plt.figure()
            shap.summary_plot(shap_values, feature_names=feature_names, plot_type=plot_type, show=False)

            title = f'SHAP Summary Plot for {class_name}'
            if title_prefix:
                title = f'{title_prefix} {title}'
            plt.title(title)

            if output_dir:
                filename = f'{title_prefix}_{class_name}_shap_summary.png' if title_prefix else f'{class_name}_shap_summary.png'
                plt.savefig(os.path.join(output_dir, filename), bbox_inches='tight')
                plt.close()
            else:
                plt.show()

# Example usage:
# Assuming `model` and `X_test` are defined, and the model is already fitted with attributes like classes_ and feature_names_.
# shap_explain = ShapExplain(model)
# raw_scores = shap_explain.predict_raw_scores(X_test)
# proba_scores = shap_explain.predict_proba(X_test)
# labels = shap_explain.predict_labels(X_test)
# shap_values = shap_explain.calculate_shap_values(X_test)
# shap_contributions = shap_explain.split_shap_values()

# # Create a Pandas Excel writer using openpyxl as the engine
# with pd.ExcelWriter('shap_explain_output.xlsx', engine='openpyxl') as writer:
#     # Convert raw_scores and labels to DataFrames and write to Excel
#     pd.DataFrame(raw_scores).to_excel(writer, sheet_name='Raw Scores')
#     pd.DataFrame(proba_scores, columns=[f"Class_{i}" for i in range(shap_explain.num_classes)]).to_excel(writer, sheet_name='Probabilities')
#     pd.DataFrame(labels, columns=['Predicted Labels']).to_excel(writer, sheet_name='Labels')

#     # Write each class's SHAP contributions to a separate sheet
#     for class_name, df in shap_contributions.items():
#         df.to_excel(writer, sheet_name=f'SHAP Values {class_name}')
```

Class `ShapExplain` ???c thi?t k? ?? h? tr? vi?c gi?i th?ch c?c m? h?nh Machine Learning s? d?ng SHAP (SHapley Additive exPlanations). Class n?y gi?p ng??i d?ng:
1. **T?nh to?n c?c gi? tr? SHAP** cho m? h?nh.
2. **T?ch ri?ng c?c ??ng g?p SHAP theo l?p (class)**.
3. **T?o c?c bi?u ?? SHAP** ?? minh h?a ?nh h??ng c?a t?ng ??c tr?ng ??n d? ?o?n c?a m? h?nh.

Class y?u c?u m? h?nh ??u v?o ph?i c? hai thu?c t?nh quan tr?ng:
- `classes_`: Danh s?ch c?c l?p trong b?i to?n ph?n lo?i.
- `feature_name_`: Danh s?ch c?c ??c tr?ng ???c s? d?ng trong m? h?nh.

### C?c ph??ng th?c ch?nh

1. **`__init__`**: 
   - Kh?i t?o class v?i m? h?nh v? ki?m tra m? h?nh c? c?c thu?c t?nh c?n thi?t (`classes_`, `feature_name_`).
   - G?n c?c gi? tr? ban ??u (s? l??ng l?p, s? l??ng ??c tr?ng, v.v.).

2. **`predict_raw_scores`**:
   - D? ?o?n ?i?m s? "th?" (raw scores) t? m? h?nh.

3. **`predict_proba`**:
   - D? ?o?n x?c su?t c?a t?ng l?p.

4. **`predict_labels`**:
   - D? ?o?n nh?n (label) b?ng c?ch ch?n l?p c? x?c su?t cao nh?t.

5. **`calculate_shap_values`**:
   - T?nh gi? tr? SHAP b?ng c?ch s? d?ng ph??ng ph?p `predict` v?i `pred_contrib=True`.

6. **`split_shap_values`**:
   - Ph?n t?ch gi? tr? SHAP theo t?ng l?p (class) th?nh c?c DataFrame ri?ng bi?t.

7. **`calculate_mean_absolute_shap_contributions`**:
   - T?nh gi? tr? trung b?nh tuy?t ??i c?a c?c ??ng g?p SHAP (mean absolute SHAP contributions) ?? hi?u ??c tr?ng n?o quan tr?ng nh?t.

8. **`plot_shap_summary`**:
   - T?o bi?u ?? SHAP (summary plot) cho t?ng l?p, l?u v?o th? m?c ho?c hi?n th? tr?c ti?p.

---


In [ ]:

import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import os

class ShapExplain:
    def __init__(self, model):
        if hasattr(model, 'classes_') and hasattr(model, 'feature_name_'):
            self.model = model
            self.raw_scores = None
            self.proba_scores = None
            self.labels = None
            self.shap_values = None
            self.shap_contributions = None

            # Initialize class attributes from the model
            self.num_classes = len(model.classes_)
            self.num_features = len(model.feature_name_)
            self.feature_names = model.feature_name_ + ["Bias"]
        else:
            raise ValueError("The provided model does not have the required attributes 'classes_' or 'feature_name_'.")

    def predict_raw_scores(self, X):
        if hasattr(self.model, 'predict'):
            self.raw_scores = self.model.predict(X, raw_score=True)
            return self.raw_scores
        else:
            raise AttributeError("The model does not support raw score predictions.")

    def predict_proba(self, X):
        if hasattr(self.model, 'predict_proba'):
            self.proba_scores = self.model.predict_proba(X)
            return self.proba_scores
        else:
            raise AttributeError("The model does not support probability predictions.")

    def predict_labels(self, X):
        self.predict_proba(X)  # Ensure probabilities are always calculated
        self.labels = [np.argmax(score) for score in self.proba_scores]
        return self.labels

    def calculate_shap_values(self, X):
        if hasattr(self.model, 'predict'):
            self.shap_values = self.model.predict(X, pred_contrib=True)
            return self.shap_values
        else:
            raise AttributeError("The model does not support SHAP value predictions.")

    def split_shap_values(self):
        if self.shap_values is None:
            raise ValueError("SHAP values have not been calculated. Call calculate_shap_values first.")
        
        shap_contributions = {}
        for class_idx in range(self.num_classes):
            start_idx = class_idx * (self.num_features + 1)
            end_idx = start_idx + (self.num_features + 1)
            shap_contributions[f"Class_{class_idx}"] = pd.DataFrame(
                self.shap_values[:, start_idx:end_idx], columns=self.feature_names
            )
        self.shap_contributions = shap_contributions
        return self.shap_contributions
    
    def calculate_mean_absolute_shap_contributions(self):
        """
        Calculate the mean absolute SHAP contributions for each feature and class.

        Returns:
            pd.DataFrame: A DataFrame where rows represent features, columns represent classes,
                        and values are the mean absolute SHAP contributions.
        """
        if self.shap_contributions is None:
            raise ValueError("SHAP contributions have not been split. Call split_shap_values first.")
        
        mean_abs_contributions = {}
        for class_name, shap_values_df in self.shap_contributions.items():
            # Exclude the "Bias" column and calculate the mean absolute value
            mean_abs_contributions[class_name] = shap_values_df.iloc[:, :-1].abs().mean(axis=0)
        
        # Combine into a DataFrame for easier interpretation
        mean_abs_df = pd.DataFrame(mean_abs_contributions)
        mean_abs_df.index.name = "Feature"
        
        return mean_abs_df

    def plot_shap_summary(self, plot_type='bar', output_dir=None, title_prefix=None):
        
        # Ensure output directory exists if specified
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)

        for class_name, shap_values_df in self.shap_contributions.items():
            shap_values = shap_values_df.iloc[:, :-1].values
            feature_names = shap_values_df.columns[:-1]  # Exclude 'Bias' from feature names

            plt.figure()
            shap.summary_plot(shap_values, feature_names=feature_names, plot_type=plot_type, show=False)

            title = f'SHAP Summary Plot for {class_name}'
            if title_prefix:
                title = f'{title_prefix} {title}'
            plt.title(title)

            if output_dir:
                filename = f'{title_prefix}_{class_name}_shap_summary.png' if title_prefix else f'{class_name}_shap_summary.png'
                plt.savefig(os.path.join(output_dir, filename), bbox_inches='tight')
                plt.close()
            else:
                plt.show()

In [ ]:
# LightGBM Predict Contrib for Multi-Class Classification

# Import necessary libraries
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Step 1: Load Dataset
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target

# Step 2: Split Dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Step 3: Train LightGBM Model
model = lgb.LGBMClassifier(verbosity = -1)
model.fit(X_train, y_train)



In [ ]:
# Example usage:
shap_explain = ShapExplain(model)
raw_scores = shap_explain.predict_raw_scores(X_test)
proba_scores = shap_explain.predict_proba(X_test)
labels = shap_explain.predict_labels(X_test)
shap_values = shap_explain.calculate_shap_values(X_test)
shap_contributions = shap_explain.split_shap_values()
mean_abs_contributions = shap_explain.calculate_mean_absolute_shap_contributions()

In [ ]:
shap_explain.plot_shap_summary()